# FLUKE Dialogue Contradiction Detection with OpenAI o3-2025-04-16 Reasoning Model

This notebook evaluates dialogue contradiction detection robustness using OpenAI's o3-2025-04-16 reasoning model with FLUKE linguistic modifications.

In [ ]:
from datasets import load_dataset
import dspy
import openai
import os
import re
import pandas as pd
import json
import random
from dotenv import load_dotenv
import glob
from scipy import stats
import time
from tqdm import tqdm

In [ ]:
load_dotenv()

In [ ]:
openai.api_key = os.getenv('OPENAI_API_KEY')
openai.organization = os.getenv('OPENAI_ORGANIZATION')

## Model Configuration

Configure o3-2025-04-16 reasoning model for dialogue understanding.

In [ ]:
# Available o3 and o1 reasoning models
REASONING_MODELS = {
    'o3-2025-04-16': 'openai/o3-2025-04-16',
    'o1-preview': 'openai/o1-preview',
    'o1-mini': 'openai/o1-mini',
    'o1': 'openai/o1',
}

# Model selection with different reasoning strategies
REASONING_CONFIGS = {
    'standard': {
        'model': 'o3-2025-04-16',
        'instruction_style': 'standard',
        'description': 'Standard reasoning approach with o3'
    },
    'detailed': {
        'model': 'o3-2025-04-16',
        'instruction_style': 'detailed',
        'description': 'Detailed step-by-step reasoning with o3'
    },
    'efficient': {
        'model': 'o1-mini',
        'instruction_style': 'concise',
        'description': 'Efficient reasoning with o1-mini'
    }
}

# Select configuration
CONFIG_NAME = 'standard'  # Change to 'detailed' or 'efficient'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]
INSTRUCTION_STYLE = config['instruction_style']

print(f"Using configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Instruction style: {INSTRUCTION_STYLE}")
print(f"Description: {config['description']}")

In [ ]:
# Configure DSPy
lm = dspy.LM(MODEL_ID)
dspy.configure(lm=lm)

## Load Dialogue Data

In [ ]:
# Load dialogue dataset
ds = pd.read_json('../data/train_dev_test_data/dialog/test.json')
ds = ds.to_dict('records')

print(f"Loaded {len(ds)} dialogue samples")

In [ ]:
samples_with_contradiction = []
samples_no_contradiction = []
for i, x in enumerate(ds):
    if x["is_contradiction"]:
        samples_with_contradiction.append((i, x))
    else:
        samples_no_contradiction.append((i, x))
print(f"Contradictions: {len(samples_with_contradiction)}, No contradictions: {len(samples_no_contradiction)}")

In [ ]:
samples = samples_with_contradiction + samples_no_contradiction
random.shuffle(samples)

label_map = {'is_contradiction': 1, 'no_contradiction': 0}

In [ ]:
def remove_space(text):
    """Clean up spacing and formatting in dialogue text."""
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        # Remove multiple spaces
        cleaned = ' '.join(line.split())
        
        # Fix spacing around punctuation
        cleaned = re.sub(r'\s+([.,!?])', r'\1', cleaned)
        cleaned = re.sub(r'([.,!?])\s+', r'\1 ', cleaned)
        
        # Fix contractions
        cleaned = re.sub(r'\s*\'\s*s\b', "'s", cleaned)
        cleaned = re.sub(r'\s*n\s*\'\s*t\b', "n't", cleaned)
        cleaned = re.sub(r'\s*\'\s*ve\b', "'ve", cleaned)
        cleaned = re.sub(r'\s*\'\s*re\b', "'re", cleaned)
        cleaned = re.sub(r'\s*\'\s*ll\b', "'ll", cleaned)
        cleaned = re.sub(r'\s*\'\s*d\b', "'d", cleaned)
        cleaned = re.sub(r'\s*\'\s*m\b', "'m", cleaned)
        
        # Fix spaces around parentheses
        cleaned = re.sub(r'\(\s+', '(', cleaned)
        cleaned = re.sub(r'\s+\)', ')', cleaned)
        
        # Remove leading/trailing whitespace
        cleaned = cleaned.strip()
        
        cleaned_lines.append(cleaned)
        
    return '\n'.join(cleaned_lines)

In [ ]:
# Function to add agent labels to dialogue
def add_agent_labels(dialogue_list):
    """Add agent 0/1 labels to each turn in the dialogue."""
    labeled_dialogue = []
    for i, turn in enumerate(dialogue_list):
        agent_label = f"agent {i % 2}: {turn}"
        labeled_dialogue.append(agent_label)
    return '\n'.join(labeled_dialogue)

# Create examples with agent labels
examples = [
    dspy.Example({ 
                  "dialogue": remove_space(add_agent_labels(r["dialogue"])), 
                  "label": label_map[r['label']]
                }).with_inputs("dialogue") 
    for i, r in samples
]

In [ ]:
example = examples[59]
for k, v in example.items():
    print(f"\n{k.upper()}:\n")
    print(v)

In [ ]:
def extract_prediction(text):
    """Extract prediction from o3 model output."""
    matches = re.findall(r'\b[0-2]\b', text)
    parsed_answer = matches[-1] if matches else ""
    return parsed_answer

In [ ]:
def eval_metric(true, prediction, trace=None):
    """Evaluate dialogue prediction."""
    pred = prediction.label
    matches = re.findall(r'\b[0-2]\b', pred)
    parsed_answer = matches[-1] if matches else ""
    return parsed_answer == str(true.label)

In [ ]:
def append_person(ds):
    """Add agent labels to dialogue turns."""
    for i, sample in enumerate(ds):
        modified_dialog = sample['dialog_context'] + [sample['modified_text']]
        original_dialog = sample['dialog_context'] + [sample['original_text']]
        for j, turn in enumerate(modified_dialog):
            turn = 'agent ' + str(j%2) + ': ' + turn
            modified_dialog[j] = turn
        for j, turn in enumerate(original_dialog):
            turn = 'agent ' + str(j%2) + ': ' + turn
            original_dialog[j] = turn
        ds[i]['original_dialog'] = '\n'.join(original_dialog)
        ds[i]['original_dialog'] = remove_space(ds[i]['original_dialog'])
        ds[i]['modified_dialog'] = '\n'.join(modified_dialog)
        ds[i]['modified_dialog'] = remove_space(ds[i]['modified_dialog'])
    return ds

# Evaluate the original test set

In [ ]:
from dspy.evaluate import Evaluate

## Dialogue Contradiction Detection with o3 Model

In [ ]:
class O3Dialogue(dspy.Signature):
    """Given a dialogue with agent labels (agent 0 and agent 1 alternating), determine if the last utterance contradicts the dialogue context. Think step by step about the conversation flow, consistency of statements, and logical coherence. Answer with 1 if it contradicts, 0 if it does not contradict."""
    dialogue = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

In [ ]:
class O3DialogueModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(O3Dialogue)

    def forward(self, dialogue):
        return self.prog(dialogue=dialogue)

In [ ]:
o3_dialogue = O3DialogueModule()

In [ ]:
# Test with a single example
pred = o3_dialogue(dialogue=example.dialogue)
print("\nDIALOGUE:\n")
print(example.dialogue)
print("\nTRUE LABEL:\n")
print(example.label)
print("\nPREDICTION:\n")
print(pred)
print(f"\nCorrect: {eval_metric(example, pred)}")

## Evaluate Original Test Set

In [ ]:
# Use subset for testing due to o3 costs and rate limits
test_examples = examples[:100]  # Adjust size as needed

print(f"Evaluating on {len(test_examples)} examples")

evaluate = Evaluate(
    devset=test_examples, 
    metric=eval_metric, 
    num_threads=1,  # Lower for o3 models
    display_progress=True, 
    display_table=10, 
    return_outputs=True, 
    return_all_scores=True
)

results = evaluate(o3_dialogue)

# Save results
items = []
for sample in results[1]:
    item = {
        'dialog': sample[0]['dialogue'],
        'label': sample[0]['label'],
        'pred': extract_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']  # Save full reasoning
    }
    items.append(item)

df_result = pd.DataFrame(data=items)
df_result.to_csv(f'results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-dialogue.csv', index=False)
print(f"Results saved to results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-dialogue.csv")
print(f"Accuracy: {results[0]:.3f}")

## Chain-of-Thought with o3 (Enhanced Reasoning)

In [ ]:
class CoTO3Dialogue(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(O3Dialogue)

    def forward(self, dialogue):
        return self.prog(dialogue=dialogue)

In [ ]:
cot_o3_dialogue = CoTO3Dialogue()
pred_cot = cot_o3_dialogue(dialogue=example.dialogue)
print("\nCHAIN-OF-THOUGHT EXAMPLE:\n")
print(f"Dialogue: {example.dialogue[:200]}...")
print("\nCOT PREDICTION:\n")
print(pred_cot)

In [ ]:
# Evaluate CoT version (optional)
evaluate_cot = Evaluate(
    devset=test_examples[:50], 
    metric=eval_metric, 
    num_threads=1, 
    display_progress=True, 
    display_table=5, 
    return_outputs=True, 
    return_all_scores=True
)

results_cot = evaluate_cot(cot_o3_dialogue)

items_cot = []
for sample in results_cot[1]:
    item = {
        'dialog': sample[0]['dialogue'],
        'label': sample[0]['label'],
        'pred': extract_prediction(sample[1]['label']),
        'reasoning': sample[1].get('reasoning', ''),
        'raw_output': sample[1]['label']
    }
    items_cot.append(item)

df_result_cot = pd.DataFrame(data=items_cot)
df_result_cot.to_csv(f'results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-cot-dialogue.csv', index=False)
print(f"CoT Accuracy: {results_cot[0]:.3f}")

# Evaluate by modification

## Without label change

In [ ]:
def evaluate_modified_set(ds, program, max_samples=50):
    """Evaluate on modified dataset with sample limit."""
    # Limit samples due to o3 cost and rate limits
    limited_ds = ds[:max_samples] if len(ds) > max_samples else ds
    
    examples = [
        dspy.Example({ 
                      "dialogue": r['modified_dialog'], 
                      "original_dialogue": r['original_dialog'],
                      "label": int(r['label']),
                      "modified_label": int(r['label'])
                    }).with_inputs("dialogue") 
        for r in limited_ds
    ]
    
    evaluate = Evaluate(
        devset=examples, 
        metric=eval_metric, 
        num_threads=1, 
        display_progress=True, 
        display_table=1, 
        return_outputs=True, 
        return_all_scores=True
    )
    
    return evaluate(program)

In [ ]:
# Recreate classes for consistency
class O3Dialogue(dspy.Signature):
    """Given a dialogue with agent labels (agent 0 and agent 1 alternating), determine if the last utterance contradicts the dialogue context. Answer with 1 if contradict, 0 if not contradict."""
    dialogue = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

class O3DialogueModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(O3Dialogue)

    def forward(self, dialogue):
        return self.prog(dialogue=dialogue)
        
o3_dialogue = O3DialogueModule()

In [ ]:
# Configure o3 model and load original predictions
lm = dspy.LM(MODEL_ID)
dspy.configure(lm=lm)

# Load original predictions for comparison
original_pred_file = f'results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-dialogue.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file, index_col=False)
    original_pred_ds['dialog'] = original_pred_ds['dialog'].apply(remove_space)
    print(f"Loaded original predictions from {original_pred_file}")
else:
    print(f"Original predictions file not found: {original_pred_file}")
    print("Please run the original evaluation first")
    original_pred_ds = None

# Get modification files (subset for testing)
json_files = glob.glob('../data/modified_data/dialogue/*_100.json')
test_modifications = ['typo_bias_100.json', 'capitalization_100.json', 'punctuation_100.json']
json_files = [f for f in json_files if any(mod in f for mod in test_modifications)]

print(f"Testing modifications: {[f.split('/')[-1] for f in json_files]}")

for json_file in json_files:
    print(f"\nProcessing: {json_file}")
    if any(x in json_file for x in ['grammatical_role', 'negation']):
        print("Skipping complex modification for now")
        continue
        
    # Load data based on file type
    if not any(x in json_file for x in ['capitalization', 'typo_bias', 'punctuation', 'grammatical_role', 'negation']):
        data = pd.read_json(json_file)[1]
        data = list(data)
    else:
        with open(json_file, 'r') as f:
            data = json.load(f)
    
    data = append_person(data)
    results_modified = evaluate_modified_set(data, o3_dialogue, max_samples=20)
    
    # Convert results to dataframe
    items = []
    for sample in results_modified[1]:
        item = {}
        modified_dialog = sample[0]['dialogue']
        original_dialog = sample[0]['original_dialogue']
        label = sample[0]['label']
        pred = sample[1]['label']
        
        original_dialog = remove_space(original_dialog)
        pred_clean = extract_prediction(pred)
        
        # Find original prediction
        original_pred = None
        if original_pred_ds is not None:
            matching_rows = original_pred_ds[original_pred_ds['dialog'] == original_dialog]
            if not matching_rows.empty:
                original_pred = matching_rows.iloc[0]['pred']
        
        item['original_dialog'] = original_dialog
        item['modified_dialog'] = modified_dialog
        item['modified_label'] = label
        item['original_label'] = label
        item['modified_pred'] = pred_clean
        item['original_pred'] = original_pred
        item['raw_output'] = pred
        items.append(item)
    
    df_result = pd.DataFrame(data=items)
    
    # Save results
    output_filename = f"results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-{json_file.split('/')[-1].replace('.json', '')}.csv"
    df_result.to_csv(output_filename, index=False)
    print(f"Saved results to: {output_filename}")
    print(f"Accuracy: {results_modified[0]:.3f}")
    
    # Add delay to respect rate limits
    time.sleep(5)

## With label change

In [ ]:
def evaluate_modified_set_with_label_change(ds, program, max_samples=50):
    """Evaluate on modified dataset where labels might change."""
    limited_ds = ds[:max_samples] if len(ds) > max_samples else ds
    
    examples = [
        dspy.Example({ 
                      "dialogue": r['modified_dialog'], 
                      "original_dialogue": r['original_dialog'],
                      "label": int(r['modified_label']) if r.get('modified_label') is not None else int(r['label']),
                      "original_label": int(r['label']),
                      "index": r.get('index', 0),
                      "type": r.get('type', None)
                    }).with_inputs("dialogue") 
        for r in limited_ds
    ]
    
    evaluate = Evaluate(
        devset=examples, 
        metric=eval_metric, 
        num_threads=1, 
        display_progress=True, 
        display_table=1, 
        return_outputs=True, 
        return_all_scores=True
    )
    
    return evaluate(program)

In [ ]:
# Test modifications that change labels
json_files_label_change = glob.glob('../data/modified_data/dialogue/*_100.json')
label_change_modifications = ['geographical_bias_100.json']
json_files_label_change = [f for f in json_files_label_change if any(mod in f for mod in label_change_modifications)]

for json_file in json_files_label_change:
    print(f"\nProcessing label-changing modification: {json_file}")
    
    with open(json_file, 'r') as f:
        raw_data = json.load(f)
    data = [dict(sample, index=idx) for idx, sample in enumerate(raw_data)]
    
    data = append_person(data)
    results_modified = evaluate_modified_set_with_label_change(data, o3_dialogue, max_samples=15)
    
    # Convert results to dataframe
    items = []
    for sample in results_modified[1]:
        item = {}
        modified_dialog = sample[0]['dialogue']
        original_dialog = sample[0]['original_dialogue']
        pred = sample[1]['label']
        
        original_dialog = remove_space(original_dialog)
        pred_clean = extract_prediction(pred)
        
        # Find original prediction
        original_pred = None
        if original_pred_ds is not None:
            matching_rows = original_pred_ds[original_pred_ds['dialog'] == original_dialog]
            if not matching_rows.empty:
                original_pred = matching_rows.iloc[0]['pred']
        
        item['original_dialog'] = original_dialog
        item['modified_dialog'] = modified_dialog
        item['modified_label'] = sample[0]['label']
        item['original_label'] = sample[0]['original_label']
        item['modified_pred'] = pred_clean
        item['original_pred'] = original_pred
        item['type'] = sample[0]['type']
        item['raw_output'] = pred
        items.append(item)
    
    df_result = pd.DataFrame(data=items)
    
    # Save results
    output_filename = f"results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-{json_file.split('/')[-1].replace('.json', '')}.csv"
    df_result.to_csv(output_filename, index=False)
    print(f"Saved results to: {output_filename}")
    print(f"Accuracy: {results_modified[0]:.3f}")
    
    time.sleep(5)

# Aggregate results

In [ ]:
from scipy import stats

In [ ]:
# Aggregate results across all modifications
result_files = glob.glob(f'results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')
aggregated_results = []

print(f"Found {len(result_files)} result files for {MODEL_NAME}-{CONFIG_NAME}")

for file in result_files:
    # Extract modification type from filename
    mod_type = file.split('-')[-1].replace('.csv', '')
    
    try:
        # Read results file
        df = pd.read_csv(file)
        
        # Handle missing columns gracefully
        if 'original_pred' not in df.columns or df['original_pred'].isna().all():
            print(f"Warning: {file} missing original predictions")
            continue
            
        # Calculate accuracies
        original_correct = (df['original_pred'] == df['original_label']).sum()
        modified_correct = (df['modified_pred'] == df['modified_label']).sum()
        total = len(df)

        if total == 0:
            continue
            
        original_acc = original_correct / total
        modified_acc = modified_correct / total
        
        # Calculate the difference
        difference = -round(original_acc - modified_acc, 2)
        
        # Calculate percentage difference
        if original_correct > 0:
            pct_difference = -round((original_correct - modified_correct) / original_correct * 100, 2)
        else:
            pct_difference = 0
        
        # Perform t-test if we have enough data
        try:
            t_stat, p_value = stats.ttest_ind(
                (df['original_pred'] == df['original_label']).astype(float),
                (df['modified_pred'] == df['modified_label']).astype(float)
            )
        except:
            p_value = None
        
        aggregated_results.append({
            'task': 'dialogue_contradiction_detection',
            'model': f'{MODEL_NAME}-{CONFIG_NAME}',
            'modification': mod_type,
            'original_res': round(original_acc, 3),
            'modified_res': round(modified_acc, 3),
            'difference': difference,
            'pct_difference': pct_difference,
            'p_value': p_value,
            'samples': total
        })
        
    except Exception as e:
        print(f"Error processing {file}: {e}")
        continue

# Create final results dataframe
if aggregated_results:
    results_df = pd.DataFrame(aggregated_results)
    
    # Sort the results
    modification_order = ['temporal_bias_100', 'geographical_bias_100', 'length_bias_100', 
                         'typo_bias_100', 'capitalization_100', 'punctuation_100', 
                         'derivation_100', 'compound_word_100', 'active_to_passive_100',
                         'grammatical_role_100', 'coordinating_conjunction_100', 
                         'concept_replacement_100', 'negation_100', 'discourse_100',
                         'sentiment_100', 'casual_100', 'dialectal_100']
    
    # Only use modifications that exist in our results
    existing_mods = results_df['modification'].unique()
    modification_order = [mod for mod in modification_order if mod in existing_mods]
    
    results_df['modification'] = pd.Categorical(results_df['modification'], categories=modification_order, ordered=True)
    results_df = results_df.sort_values(by='modification')

    # Calculate averages across all modifications
    avg_original = results_df['original_res'].mean()
    avg_modified = results_df['modified_res'].mean()
    avg_difference = avg_original - avg_modified
    avg_pct_difference = results_df['pct_difference'].mean()

    # Add averages as a new row
    avg_row = {
        'task': 'dialogue_contradiction_detection',
        'model': f'{MODEL_NAME}-{CONFIG_NAME}',
        'modification': 'average',
        'original_res': round(avg_original, 3),
        'modified_res': round(avg_modified, 3),
        'difference': -round(avg_difference, 3),
        'pct_difference': round(avg_pct_difference, 2),
        'p_value': None,
        'samples': results_df['samples'].sum()
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([avg_row])], ignore_index=True)

    print(f"\n{MODEL_NAME}-{CONFIG_NAME} Results Summary:")
    print(results_df[['modification', 'original_res', 'modified_res', 'difference', 'samples']])

    # Save aggregated results
    results_df.to_csv(f'results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-DP.csv', index=False)
    print(f"\nAggregated results saved to: results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-DP.csv")

    # Apply styling to highlight performance drops
    def highlight_drops_and_significance(row):
        colors = [''] * len(row)
        if row['original_res'] > row['modified_res']:
            colors = ['background-color: red'] * len(row)
            # If p-value < 0.05, add bold text
            if 'p_value' in row and row['p_value'] is not None and row['p_value'] < 0.05:
                colors = ['background-color: red; font-weight: bold'] * len(row)
        return colors

    styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
    display(styled_df)
    
else:
    print("No results found to aggregate")

## Model Comparison and Analysis

In [ ]:
# Compare with other models if available
comparison_files = {
    'GPT-4o': 'results/dialogue/gpt4o-0shot-dialogue.csv',
    'Claude-3.5': 'results/dialogue/claude-3-5-sonnet-0shot-dialogue.csv',
    'Mixtral-8x22B': 'results/dialogue/mixtral-8x22b-0shot-dialogue.csv',
    f'{MODEL_NAME}-{CONFIG_NAME}': f'results/dialogue/{MODEL_NAME}-{CONFIG_NAME}-0shot-dialogue.csv'
}

model_accuracies = {}
for model_name, file_path in comparison_files.items():
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            # Handle different column names
            pred_col = 'pred' if 'pred' in df.columns else 'prediction'
            if pred_col in df.columns and 'label' in df.columns:
                accuracy = (df[pred_col] == df['label']).mean()
                model_accuracies[model_name] = accuracy
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    else:
        print(f"File not found: {file_path}")

# Display comparison
if model_accuracies:
    comparison_df = pd.DataFrame([
        {'Model': model, 'Accuracy': acc, 'Performance': f"{acc:.1%}"} 
        for model, acc in model_accuracies.items()
    ])
    comparison_df = comparison_df.sort_values('Accuracy', ascending=False)
    
    print("\nModel Comparison on Dialogue Contradiction Detection:")
    print(comparison_df)

    # Highlight o3 performance
    o3_model_key = f'{MODEL_NAME}-{CONFIG_NAME}'
    o3_performance = model_accuracies.get(o3_model_key, 0)
    print(f"\n{o3_model_key} Accuracy: {o3_performance:.3f} ({o3_performance:.1%})")
    
    if len(model_accuracies) > 1:
        other_models = [acc for model, acc in model_accuracies.items() if model != o3_model_key]
        if other_models:
            avg_others = sum(other_models) / len(other_models)
            improvement = o3_performance - avg_others
            print(f"Average of other models: {avg_others:.3f} ({avg_others:.1%})")
            print(f"Performance difference: {improvement:+.3f} ({improvement:+.1%})")

    # Style the dataframe
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]

    styled_comparison = comparison_df.style.apply(highlight_max, subset=['Accuracy'])
    display(styled_comparison)
else:
    print("No model comparison data available")

## o3 Reasoning Analysis

In [ ]:
# Analyze o3 reasoning quality (if raw outputs are available)
if 'raw_output' in df_result.columns and not df_result.empty:
    print("Sample o3 Reasoning for Dialogue Contradiction Detection:")
    print("=" * 60)
    
    for i, (idx, row) in enumerate(df_result.head(3).iterrows()):
        print(f"\nExample {i+1}:")
        print(f"Dialogue: {row['dialog'][:200]}{'...' if len(row['dialog']) > 200 else ''}")
        print(f"True Label: {row['label']} ({'Contradiction' if row['label'] == 1 else 'No contradiction'})")
        print(f"Prediction: {row['pred']} ({'Contradiction' if row['pred'] == '1' else 'No contradiction'})")
        print(f"Reasoning: {row['raw_output'][:600]}{'...' if len(str(row['raw_output'])) > 600 else ''}")
        print("-" * 50)

# Summary statistics
print(f"\n{MODEL_NAME}-{CONFIG_NAME} Evaluation Summary:")
print("=" * 60)

if 'results' in locals():
    print(f"Base accuracy on dialogue contradiction: {results[0]:.3f} ({results[0]:.1%})")

if 'aggregated_results' in locals() and aggregated_results:
    avg_robustness = sum([r['difference'] for r in aggregated_results if r['difference'] is not None]) / len([r for r in aggregated_results if r['difference'] is not None])
    print(f"Average robustness impact: {avg_robustness:+.3f}")
    print(f"Modifications tested: {len(aggregated_results)}")

print(f"\nKey insights with {MODEL_NAME}:")
print(f"- Enhanced reasoning model for dialogue understanding")
print(f"- Detailed reasoning traces show conversation flow analysis")
print(f"- Performance on linguistic robustness varies by modification complexity")
print(f"- Advanced reasoning helps with multi-turn dialogue coherence")

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE Dialogue Evaluation with {MODEL_NAME} Complete!")
print(f"{'='*60}")
print(f"Configuration: {config['description']}")
print(f"Files saved in results/dialogue/ with prefix '{MODEL_NAME}-{CONFIG_NAME}-'")
print(f"\nNext steps:")
print(f"1. Review reasoning traces for dialogue understanding strategies")
print(f"2. Compare robustness with other models on different modifications")
print(f"3. Analyze which linguistic changes most challenge {MODEL_NAME}")
print(f"4. Consider different reasoning configurations (detailed vs standard)")
print(f"5. Evaluate cost-benefit for dialogue understanding tasks")